# Train Type-2 Physics SFT Model — Option B: One Adapter, Explicit Task Modes

This notebook trains one Qwen LoRA adapter using the original base model default `Qwen/Qwen2.5-7B-Instruct`, but every sample is labeled with a clear task mode:

- `generate_code`: question → executable Python/SymPy solver JSON
- `explain_from_result`: question + sandbox result → final answer/explanation JSON
- `qualitative_answer`: theory/qualitative question → final answer/explanation JSON, no code

The train/test split is done at the **problem ID level** before creating task-mode rows, so the model cannot see one task mode of a problem in train and another task mode of the same problem in test.


**Base model default restored from the original uploaded notebook:** `Qwen/Qwen2.5-7B-Instruct`. The Option B changes affect task formatting/evaluation only; they do not change the base model.


In [ ]:
import os
import sys
import json
import torch
from pathlib import Path
import argparse
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    set_seed
)
from peft import LoraConfig, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login

# ---- AMD MI300X / ROCm detection ----
def detect_attn_implementation():
    """Return the best available attention implementation for the current GPU."""
    if not torch.cuda.is_available():
        return "eager"
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else ""
    is_amd = any(x in gpu_name.lower() for x in ["amd", "instinct", "radeon", "mi300", "mi250", "mi210"])
    if is_amd:
        # ROCm may not have flash_attention_2 compiled; try it, fall back to sdpa
        try:
            from flash_attn import flash_attn_func
            print(f"[AMD/ROCm] flash_attention_2 available for {gpu_name}")
            return "flash_attention_2"
        except (ImportError, OSError):
            print(f"[AMD/ROCm] flash_attention_2 not available; falling back to sdpa for {gpu_name}")
            return "sdpa"
    return "flash_attention_2"

ATTN_IMPL = detect_attn_implementation()

# Safe repo cloning and directory changing
repo_url = "https://github.com/NguyenAn05/exact-2026.git"
repo_dir = "exact-2026"

if Path(os.getcwd()).name == repo_dir:
    print(f"Already inside the repository directory: {os.getcwd()}")
else:
    if not os.path.exists(repo_dir):
        print(f"Cloning repository from {repo_url}...")
        os.system(f"git clone {repo_url}")
    os.chdir(repo_dir)
    print(f"Changed working directory to: {os.getcwd()}")

print(f"Current working directory: {os.getcwd()}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Attention implementation: {ATTN_IMPL}")

In [ ]:
def parse_args():
    parser = argparse.ArgumentParser(description="Option-B SFT Training for Type-2 physics solver: one adapter, explicit task modes.")
    parser.add_argument("--model_id", type=str, default="Qwen/Qwen2.5-7B-Instruct")
    parser.add_argument("--gen_dataset_id", type=str, default="not-a-real-ai-guy/exact_2026_type2_generator")
    parser.add_argument("--exp_dataset_id", type=str, default="not-a-real-ai-guy/exact_2026_type2_explainer")
    parser.add_argument("--output_dir", type=str, default="models/qwen2.5-type2-option-b-modes-lora")
    parser.add_argument("--hub_model_id", type=str, default="not-a-real-ai-guy/qwen2.5-type2-option-b-modes-lora")
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--grad_accum", type=int, default=2)
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--lr", type=float, default=1e-4, help="LoRA SFT usually benefits from a higher LR than full finetuning; sweep 5e-5, 1e-4, 2e-4.")
    parser.add_argument("--max_seq_len", type=int, default=4096)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--max_steps", type=int, default=-1)
    parser.add_argument("--test_split", type=float, default=0.2)
    parser.add_argument("--save_steps", type=int, default=50)
    parser.add_argument("--include_invalid_as_qualitative", action="store_true", help="Only use invalid/non-runner problems as qualitative when they look explicitly theory/conceptual. Use this flag to include all invalid problems as qualitative answer tasks if explainer labels exist.")

    # Check if running in a Jupyter notebook/IPython session
    is_ipython = False
    try:
        from IPython import get_ipython
        if get_ipython() is not None:
            is_ipython = True
    except ImportError:
        pass

    if is_ipython or any('ipykernel' in arg for arg in sys.argv):
        return parser.parse_args(args=[])
    else:
        return parser.parse_args()


In [ ]:
# =========================================================================
# 🔐 Secure HF Token Input
# =========================================================================
# Your token is never stored in the notebook — it's read at runtime via a hidden prompt.
# If HF_TOKEN env var is already set, it will be used instead.

import getpass
import os

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")

if hf_token:
    print("✓ HF_TOKEN found in environment — using that.")
else:
    print("No HF_TOKEN environment variable set.")
    print("Enter your HuggingFace token (input will be hidden):")
    hf_token = getpass.getpass("HF Token: ")
    if not hf_token:
        raise RuntimeError("No HF token provided. Set HF_TOKEN env var or enter at prompt.")
    print("✓ Token received (hidden).")

login(token=hf_token)
print("✓ Logged in to HuggingFace Hub.")
# Clear token from memory — login() stores it, we don't need the variable anymore
del hf_token

In [ ]:
# =========================================================================
# Data Loading & Merging — Option B: explicit task-mode rows
# =========================================================================

args = parse_args()
set_seed(args.seed)

import random
import re
from collections import Counter, defaultdict

# ---- Load both datasets from HuggingFace Hub ----
print(f"Loading generator dataset: {args.gen_dataset_id}...")
gen_ds = load_dataset(args.gen_dataset_id)
gen_split = gen_ds["train"] if "train" in gen_ds else list(gen_ds.values())[0]
print(f"  Generator records: {len(gen_split)}")

print(f"Loading explainer dataset: {args.exp_dataset_id}...")
exp_ds = load_dataset(args.exp_dataset_id)
exp_split = exp_ds["train"] if "train" in exp_ds else list(exp_ds.values())[0]
print(f"  Explainer records: {len(exp_split)}")

# ---- Mode-specific system prompts ----
# Keep these short and explicit. The model benefits from a stable mode tag, regardless of size.
MODE_SYSTEM_PROMPTS = {
    "generate_code": (
        "You are a Type-2 physics solver in TASK_MODE=generate_code. "
        "Generate only safe executable Python/SymPy code for electric circuits, electrostatics, magnetism, and related physics problems. "
        "Return JSON only: {\"code\": \"<python code>\"}. "
        "The code may use only Python stdlib, json, math, fractions, decimal, and sympy. "
        "Define solve() returning a dict with keys answer, unit, kind, and derived. "
        "Print exactly one JSON object with print(json.dumps(solve(), ensure_ascii=False))."
    ),
    "explain_from_result": (
        "You are a Type-2 physics solver in TASK_MODE=explain_from_result. "
        "Given the original question and a sandbox execution result, produce the final answer and a concise checkable explanation. "
        "Do not generate code. Return JSON only with keys answer, explanation, cot, premises, confidence."
    ),
    "qualitative_answer": (
        "You are a Type-2 physics solver in TASK_MODE=qualitative_answer. "
        "For conceptual or theory-only physics questions, answer directly without code. "
        "Return JSON only with keys answer, explanation, cot, premises, confidence."
    ),
}

ANSWER_JSON_SCHEMA = (
    '{"answer": "<final answer with unit if applicable>", '
    '"explanation": "<concise explanation>", '
    '"cot": ["Step 1: ...", "Step 2: ..."], '
    '"premises": ["<formula or given value>"], '
    '"confidence": 0.0}'
)

CODE_JSON_SCHEMA = '{"code": "<python code>"}'


def _ensure_dict(x):
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        try:
            y = json.loads(x)
            return y if isinstance(y, dict) else {}
        except Exception:
            return {}
    return {}


def _content_to_str(x):
    if isinstance(x, str):
        return x
    return json.dumps(x, ensure_ascii=False)


def _last_assistant_content(example):
    msgs = example.get("messages", []) or []
    for m in reversed(msgs):
        if isinstance(m, dict) and m.get("role") == "assistant":
            return _content_to_str(m.get("content", ""))
    return ""


def _extract_question(example, meta):
    q = meta.get("question") or meta.get("problem") or meta.get("prompt") or ""
    if q:
        return str(q).strip()

    msgs = example.get("messages", []) or []
    for m in msgs:
        if not isinstance(m, dict) or m.get("role") != "user":
            continue
        content = _content_to_str(m.get("content", ""))
        # Common format: "Question: ..."
        for line in content.splitlines():
            if line.strip().lower().startswith("question:"):
                return line.split(":", 1)[1].strip()
        # Fall back to the first substantial user content.
        if len(content.strip()) > 20:
            return content.strip()
    return ""


def _has_number(x):
    return bool(re.search(r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?", str(x or "")))


def _looks_qualitative(gen_meta, exp_meta, runner_result):
    blob = " ".join(str(x or "") for x in [
        gen_meta.get("kind"), gen_meta.get("task_kind"), gen_meta.get("runner_status"),
        exp_meta.get("kind"), exp_meta.get("task_kind"),
        runner_result.get("kind") if isinstance(runner_result, dict) else "",
        gen_meta.get("skip_reason"), gen_meta.get("reason"), exp_meta.get("reason"),
    ]).lower()
    qualitative_markers = [
        "qualitative", "theory", "conceptual", "non-numeric", "nonnumeric",
        "no code", "no_code", "code not required", "not computable", "explain only",
    ]
    if any(k in blob for k in qualitative_markers):
        return True

    gold_answer = gen_meta.get("gold_answer") or exp_meta.get("gold_answer") or gen_meta.get("answer") or exp_meta.get("answer") or ""
    gold_unit = gen_meta.get("gold_unit") or exp_meta.get("gold_unit") or gen_meta.get("unit") or exp_meta.get("unit") or ""
    # A text-only gold answer with no numeric value is usually a conceptual/theory label.
    if gold_answer and not _has_number(gold_answer) and not str(gold_unit).strip():
        return True
    return False


def build_generate_code_messages(pid, question):
    return [
        {"role": "system", "content": MODE_SYSTEM_PROMPTS["generate_code"]},
        {"role": "user", "content": (
            "TASK_MODE=generate_code\n"
            f"Problem ID: {pid}\n"
            f"Question: {question}\n\n"
            f"Return JSON only: {CODE_JSON_SCHEMA}"
        )},
    ]


def build_explain_from_result_messages(pid, question, runner_result):
    return [
        {"role": "system", "content": MODE_SYSTEM_PROMPTS["explain_from_result"]},
        {"role": "user", "content": (
            "TASK_MODE=explain_from_result\n"
            f"Problem ID: {pid}\n"
            f"Question: {question}\n\n"
            "Code execution result JSON:\n"
            f"{json.dumps(runner_result, ensure_ascii=False)}\n\n"
            f"Return JSON only: {ANSWER_JSON_SCHEMA}"
        )},
    ]


def build_qualitative_answer_messages(pid, question):
    return [
        {"role": "system", "content": MODE_SYSTEM_PROMPTS["qualitative_answer"]},
        {"role": "user", "content": (
            "TASK_MODE=qualitative_answer\n"
            f"Problem ID: {pid}\n"
            f"Question: {question}\n\n"
            "This is a conceptual/theory-only physics question. Do not generate code.\n"
            f"Return JSON only: {ANSWER_JSON_SCHEMA}"
        )},
    ]


# ---- Build index maps by problem ID ----
gen_by_id = {example["id"]: example for example in gen_split}
exp_by_id = {example["id"]: example for example in exp_split}
matched_ids = sorted(set(gen_by_id.keys()) & set(exp_by_id.keys()))

print("\nMerge statistics:")
print(f"  Total generator records : {len(gen_split)}")
print(f"  Total explainer records : {len(exp_split)}")
print(f"  Matched pairs           : {len(matched_ids)}")
print(f"  Skipped (gen-only)      : {len(gen_by_id) - len(matched_ids)}")
print(f"  Skipped (exp-only)      : {len(exp_by_id) - len(matched_ids)}")

# ---- Convert matched problems into problem-level records first ----
# Important: split by problem_id BEFORE expanding to task-mode rows.
problem_records = []
skip_reasons = Counter()

for pid in matched_ids:
    gen_example = gen_by_id[pid]
    exp_example = exp_by_id[pid]

    gen_meta = _ensure_dict(gen_example.get("metadata", {}))
    exp_meta = _ensure_dict(exp_example.get("metadata", {}))
    validation = _ensure_dict(gen_meta.get("validation", {}))
    runner_result = _ensure_dict(gen_meta.get("runner_result", {}))

    question = _extract_question(gen_example, gen_meta) or _extract_question(exp_example, exp_meta)
    assistant_code = _last_assistant_content(gen_example)
    assistant_explanation = _last_assistant_content(exp_example)

    runner_ok = (gen_meta.get("runner_status") == "ok") and bool(validation.get("valid", True))
    qualitative = _looks_qualitative(gen_meta, exp_meta, runner_result)

    if not question:
        skip_reasons["missing_question"] += 1
        continue
    if not assistant_explanation:
        skip_reasons["missing_explainer_label"] += 1
        continue

    if runner_ok and assistant_code:
        eval_kind = "computable"
    elif qualitative or args.include_invalid_as_qualitative:
        eval_kind = "qualitative"
    else:
        skip_reasons["not_runner_ok_and_not_qualitative"] += 1
        continue

    problem_records.append({
        "id": pid,
        "question": question,
        "eval_kind": eval_kind,
        "assistant_code": assistant_code,
        "assistant_explanation": assistant_explanation,
        "runner_result": runner_result,
        "gold_answer": gen_meta.get("gold_answer", exp_meta.get("gold_answer", gen_meta.get("answer", exp_meta.get("answer", "")))),
        "gold_unit": gen_meta.get("gold_unit", exp_meta.get("gold_unit", gen_meta.get("unit", exp_meta.get("unit", "")))),
    })

print("\nProblem-level filtering:")
print(f"  Kept problems           : {len(problem_records)}")
print(f"  Skipped problems        : {sum(skip_reasons.values())}")
for reason, count in skip_reasons.most_common():
    print(f"    - {reason}: {count}")
print(f"  Problem kinds           : {dict(Counter(r['eval_kind'] for r in problem_records))}")

# ---- Problem-level train/test split ----
problem_ids = [r["id"] for r in problem_records]
rng = random.Random(args.seed)
rng.shuffle(problem_ids)

if len(problem_ids) >= 10 and args.test_split > 0:
    n_test = max(1, int(round(len(problem_ids) * args.test_split)))
    test_ids = set(problem_ids[:n_test])
    train_ids = set(problem_ids[n_test:])
else:
    test_ids = set()
    train_ids = set(problem_ids)

assert train_ids.isdisjoint(test_ids), "Train/test leakage: overlapping problem IDs found!"

# ---- Expand each problem into explicit task-mode SFT rows ----
def make_task_rows(record):
    pid = record["id"]
    q = record["question"]
    rows = []
    if record["eval_kind"] == "computable":
        rows.append({
            "id": f"{pid}::generate_code",
            "problem_id": pid,
            "task_mode": "generate_code",
            "messages": build_generate_code_messages(pid, q) + [
                {"role": "assistant", "content": record["assistant_code"]}
            ],
        })
        rows.append({
            "id": f"{pid}::explain_from_result",
            "problem_id": pid,
            "task_mode": "explain_from_result",
            "messages": build_explain_from_result_messages(pid, q, record["runner_result"]) + [
                {"role": "assistant", "content": record["assistant_explanation"]}
            ],
        })
    else:
        rows.append({
            "id": f"{pid}::qualitative_answer",
            "problem_id": pid,
            "task_mode": "qualitative_answer",
            "messages": build_qualitative_answer_messages(pid, q) + [
                {"role": "assistant", "content": record["assistant_explanation"]}
            ],
        })
    return rows

train_rows, test_rows = [], []
for record in problem_records:
    if record["id"] in train_ids:
        train_rows.extend(make_task_rows(record))
    elif record["id"] in test_ids:
        test_rows.extend(make_task_rows(record))

# Sanity check: no problem appears in both train and test after task expansion.
train_problem_ids = {row["problem_id"] for row in train_rows}
test_problem_ids = {row["problem_id"] for row in test_rows}
assert train_problem_ids.isdisjoint(test_problem_ids), "Task-level leakage: same problem appears in train and test!"

if test_rows:
    dataset = DatasetDict({
        "train": Dataset.from_list(train_rows),
        "test": Dataset.from_list(test_rows),
    })
else:
    dataset = DatasetDict({"train": Dataset.from_list(train_rows)})

print("\nFinal task-mode dataset:")
print(f"  Train task rows         : {len(dataset['train'])}")
print(f"  Test task rows          : {len(dataset['test']) if 'test' in dataset else 0}")
print(f"  Train problems          : {len(train_problem_ids)}")
print(f"  Test problems           : {len(test_problem_ids)}")
print(f"  Train task mix          : {dict(Counter(dataset['train']['task_mode']))}")
if "test" in dataset:
    print(f"  Test task mix           : {dict(Counter(dataset['test']['task_mode']))}")

# ---- Held-out problem-level eval lookup ----
eval_lookup = {}
for record in problem_records:
    if test_ids and record["id"] not in test_ids:
        continue
    eval_lookup[record["id"]] = {
        "question": record["question"],
        "gold_answer": record["gold_answer"],
        "gold_unit": record["gold_unit"],
        "eval_kind": record["eval_kind"],
    }

print(f"\nEval lookup created from held-out problems only: {len(eval_lookup)} entries.")

# ---- Print sample rows for inspection ----
print("\n" + "=" * 70)
print("SAMPLE OPTION-B TASK ROWS")
print("=" * 70)
for sample in train_rows[:3]:
    print(f"Sample ID: {sample['id']} | task_mode={sample['task_mode']}")
    for i, msg in enumerate(sample["messages"]):
        preview = msg["content"][:180].replace("\n", "\\n")
        print(f"  Turn {i} [{msg['role']}]: {preview}...")
    print("-" * 70)

print("\nDataset ready with a 'messages' column. TRL will apply the chat template and assistant-only masking.")


In [ ]:
# =========================================================================
# Tokenizer Setup
# =========================================================================

print(f"Loading tokenizer for {args.model_id}...")
tokenizer = AutoTokenizer.from_pretrained(args.model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# No manual prompt/completion formatting needed.
# Dataset rows use the TRL conversational `messages` format.
# Option B relies on the explicit TASK_MODE tag inside each user prompt.

print("Tokenizer loaded. Dataset has 'messages' column — TRL handles chat templating.")
print(f"  pad_token: {tokenizer.pad_token}")
print(f"  padding_side: {tokenizer.padding_side}")
print("  task modes:", sorted(set(dataset['train']['task_mode'])))


In [ ]:
# =========================================================================
# Load Base Model + Configure LoRA (same math as logic_train_sft.ipynb)
# =========================================================================

print(f"Loading base model {args.model_id} in bfloat16 with attn_implementation={ATTN_IMPL}...")
model = AutoModelForCausalLM.from_pretrained(
    args.model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation=ATTN_IMPL
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

# LoRA Configuration (identical to logic notebook)
peft_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

print(f"LoRA config: r={peft_config.r}, alpha={peft_config.lora_alpha}, "
      f"target_modules={peft_config.target_modules}")

In [ ]:
# =========================================================================
# SFTConfig: version-compatible config + assistant-only masking
# =========================================================================

# IMPORTANT:
# This dataset uses a single `messages` column (conversational format), not
# prompt/completion columns. For modern TRL, assistant_only_loss=True is the
# correct masking option for "train only on assistant turns".
#
# TRL's SFTConfig API changed across versions:
# - newer TRL commonly uses max_length
# - some older examples used max_seq_length
# So we build SFTConfig by checking which arguments your installed TRL accepts.
import inspect

use_steps = args.max_steps > 0
sft_params = set(inspect.signature(SFTConfig.__init__).parameters.keys())

base_sft_kwargs = dict(
    output_dir=args.output_dir,
    per_device_train_batch_size=args.batch_size,
    gradient_accumulation_steps=args.grad_accum,
    learning_rate=args.lr,
    num_train_epochs=args.epochs,
    max_steps=args.max_steps,
    bf16=torch.cuda.is_available(),
    fp16=False,
    logging_steps=10,
    eval_strategy="steps" if "test" in dataset else "no",
    eval_steps=args.save_steps if "test" in dataset else None,
    save_strategy="steps",
    save_steps=args.save_steps,
    push_to_hub=True,
    hub_model_id=args.hub_model_id,
    hub_strategy="checkpoint",
    hub_private_repo=True,
    load_best_model_at_end=True if "test" in dataset else False,
    metric_for_best_model="eval_loss" if "test" in dataset else None,
    greater_is_better=False,
    save_total_limit=3,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="adamw_torch_fused",
    report_to="none",
    gradient_checkpointing=True,
    ddp_find_unused_parameters=False,
)


def build_sft_config(raw_kwargs, requested_max_len):
    """Create SFTConfig while adapting to the installed TRL/Transformers version."""
    params = set(inspect.signature(SFTConfig.__init__).parameters.keys())
    kwargs = dict(raw_kwargs)

    # TRL version compatibility: max_seq_length -> max_length.
    if "max_seq_length" in params:
        kwargs["max_seq_length"] = requested_max_len
        length_key = "max_seq_length"
    elif "max_length" in params:
        kwargs["max_length"] = requested_max_len
        length_key = "max_length"
    else:
        length_key = None
        print("[WARN] This TRL SFTConfig exposes neither max_seq_length nor max_length; using TRL default length.")

    # Transformers compatibility: eval_strategy was named evaluation_strategy in older versions.
    if "eval_strategy" not in params and "evaluation_strategy" in params and "eval_strategy" in kwargs:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")

    # Qwen chat templates terminate assistant turns with <|im_end|>.
    # Setting this prevents the model from learning/using a mismatched EOS token.
    if "eos_token" in params:
        kwargs["eos_token"] = "<|im_end|>"

    if "assistant_only_loss" in params:
        kwargs["assistant_only_loss"] = True
        loss_mask = "assistant_only_loss=True"
    else:
        # Older TRL fallback. Prefer upgrading TRL if this branch is used.
        loss_mask = "assistant_only_loss unavailable"
        print("[WARN] This TRL version has no assistant_only_loss. Upgrade TRL for correct messages masking.")
        if "completion_only_loss" in params:
            kwargs["completion_only_loss"] = True
            loss_mask = "completion_only_loss=True fallback"

    # Drop arguments not supported by this exact installed SFTConfig.
    filtered_kwargs = {k: v for k, v in kwargs.items() if k in params}
    dropped = sorted(set(kwargs) - set(filtered_kwargs))
    if dropped:
        print("[WARN] Dropped unsupported SFTConfig args for this TRL version:", dropped)

    return SFTConfig(**filtered_kwargs), filtered_kwargs, length_key, loss_mask


# If fused AdamW is not supported on your ROCm/PyTorch build, fall back to adamw_torch.
try:
    sft_config, sft_kwargs, length_key, loss_mask = build_sft_config(base_sft_kwargs, args.max_seq_len)
except Exception as e:
    if base_sft_kwargs.get("optim") == "adamw_torch_fused":
        print(f"[WARN] Failed with adamw_torch_fused: {e}")
        print("[WARN] Falling back to adamw_torch.")
        base_sft_kwargs["optim"] = "adamw_torch"
        sft_config, sft_kwargs, length_key, loss_mask = build_sft_config(base_sft_kwargs, args.max_seq_len)
    else:
        raise

print(f"Training config: batch_size={args.batch_size}, grad_accum={args.grad_accum}, "
      f"lr={args.lr}, epochs={args.epochs}, requested_max_seq_len={args.max_seq_len}")
print(f"TRL length argument used: {length_key}")
print(f"Loss mask: {loss_mask}")
print(f"EOS token configured: {sft_kwargs.get('eos_token', tokenizer.eos_token)}")
print(f"Optimizer configured: {sft_kwargs.get('optim')}")


In [ ]:
# =========================================================================
# Initialize SFTTrainer and Run Training
# =========================================================================

print("Initializing SFTTrainer with Option-B task-mode messages dataset...")
eval_dataset = dataset.get("test") if "test" in dataset else None

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
    args=sft_config,
)

# Run SFT Training
print("\n--- Starting SFT Training ---")
trainer.train()

# Save the final adapter locally
print(f"\nSaving final LoRA adapter locally to {args.output_dir}...")
trainer.model.save_pretrained(args.output_dir)
tokenizer.save_pretrained(args.output_dir)

# Final push to HF Hub
print("\nPushing best LoRA adapter to Hugging Face Hub...")
trainer.push_to_hub()

print("Training and HF uploading complete!")

## Evaluation (Optional)

Run the trained model end-to-end: question → code → sandbox → explanation → answer.
This section demonstrates the full single-model pipeline.

In [ ]:
# =========================================================================
# Evaluation: Load Trained Model + Define Safer Sandbox + Option-B Solve Pipeline
# =========================================================================

import re
import math
import sympy as sp
from io import StringIO
import tempfile
import subprocess
import textwrap
import ast

print("Loading trained model for evaluation...")

# Load base model and apply LoRA adapter.
# Prefer local args.output_dir immediately after training; fall back to hub id.
eval_base_model = AutoModelForCausalLM.from_pretrained(
    args.model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation=ATTN_IMPL,
)
adapter_path = args.output_dir if os.path.exists(args.output_dir) else args.hub_model_id
eval_model = PeftModel.from_pretrained(eval_base_model, adapter_path)
eval_model.eval()

eval_tokenizer = AutoTokenizer.from_pretrained(args.model_id, trust_remote_code=True)
eval_tokenizer.pad_token = eval_tokenizer.eos_token
eval_tokenizer.padding_side = "right"

EOS_TOKEN_ID = eval_tokenizer.convert_tokens_to_ids("<|im_end|>")
if EOS_TOKEN_ID is None or EOS_TOKEN_ID < 0:
    EOS_TOKEN_ID = eval_tokenizer.eos_token_id

# ---- Robust JSON extraction ----
def extract_json_object(text: str) -> dict:
    """Best-effort extraction of the first valid JSON object from a model response."""
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    fenced = re.findall(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.S)
    for block in fenced:
        try:
            return json.loads(block)
        except Exception:
            continue

    start_positions = [i for i, ch in enumerate(text) if ch == "{"]
    for start in start_positions:
        depth = 0
        in_str = False
        escape = False
        for i in range(start, len(text)):
            ch = text[i]
            if in_str:
                if escape:
                    escape = False
                elif ch == "\\":
                    escape = True
                elif ch == '"':
                    in_str = False
            else:
                if ch == '"':
                    in_str = True
                elif ch == "{":
                    depth += 1
                elif ch == "}":
                    depth -= 1
                    if depth == 0:
                        candidate = text[start:i+1]
                        try:
                            return json.loads(candidate)
                        except Exception:
                            break
    raise ValueError("No valid JSON object found")

# ---- Safer Python/SymPy sandbox ----
ALLOWED_IMPORTS = {"json", "math", "sympy", "fractions", "decimal"}
BANNED_CALLS = {"open", "exec", "eval", "compile", "__import__", "input", "breakpoint"}


def validate_code_ast(code: str) -> None:
    """Reject obviously unsafe or out-of-scope generated code before execution."""
    tree = ast.parse(code)
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            if isinstance(node, ast.Import):
                names = [alias.name.split(".")[0] for alias in node.names]
            else:
                names = [(node.module or "").split(".")[0]]
            for name in names:
                if name not in ALLOWED_IMPORTS:
                    raise ValueError(f"Disallowed import: {name}")
        if isinstance(node, ast.Call):
            func = node.func
            name = None
            if isinstance(func, ast.Name):
                name = func.id
            elif isinstance(func, ast.Attribute):
                name = func.attr
            if name in BANNED_CALLS:
                raise ValueError(f"Disallowed call: {name}")


def run_sandbox(code: str, timeout_s: float = 3.0) -> dict:
    """
    Execute generated Python/SymPy code in a subprocess with a timeout.
    Returns either {"status": "ok", "result": dict} or {"status": "error", ...}.
    """
    if not code or not code.strip():
        return {"status": "error", "error": "Empty generated code"}

    try:
        validate_code_ast(code)
    except Exception as e:
        return {"status": "error", "error": f"AST validation failed: {e}"}

    wrapper = f"""
import json, math
import sympy as sp
sympy = sp

{code}

# If code did not print anything but defines solve(), call it explicitly.
if "solve" in globals() and callable(globals()["solve"]):
    _out = solve()
    print(json.dumps(_out, ensure_ascii=False))
"""
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False, encoding="utf-8") as f:
        f.write(wrapper)
        tmp_path = f.name

    try:
        proc = subprocess.run(
            [sys.executable, tmp_path],
            capture_output=True,
            text=True,
            timeout=timeout_s,
        )
        if proc.returncode != 0:
            return {
                "status": "error",
                "error": "Sandbox process failed",
                "stderr": proc.stderr[-2000:],
                "stdout": proc.stdout[-2000:],
            }

        lines = [ln.strip() for ln in proc.stdout.splitlines() if ln.strip()]
        if not lines:
            return {"status": "error", "error": "No output produced by code"}

        for line in reversed(lines):
            try:
                parsed = json.loads(line)
                if isinstance(parsed, dict):
                    return {"status": "ok", "result": parsed}
            except Exception:
                continue

        return {
            "status": "error",
            "error": "No JSON object found in stdout",
            "stdout": proc.stdout[-2000:],
        }
    except subprocess.TimeoutExpired:
        return {"status": "error", "error": f"Sandbox timeout after {timeout_s}s"}
    except Exception as e:
        return {"status": "error", "error": f"Sandbox error: {e}"}
    finally:
        try:
            os.remove(tmp_path)
        except Exception:
            pass


def generate_json(messages, max_new_tokens=1024):
    """Deterministic JSON generation for evaluation."""
    inputs = eval_tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(eval_model.device)

    with torch.no_grad():
        outputs = eval_model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            eos_token_id=EOS_TOKEN_ID,
            pad_token_id=eval_tokenizer.pad_token_id,
        )

    text = eval_tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    parsed = extract_json_object(text)
    return parsed, text

# ---- Option-B solve pipeline ----
def solve_one(question: str, problem_id: str = "test", eval_kind: str = "computable") -> dict:
    """
    Option B evaluation pipeline:
    - computable: TASK_MODE=generate_code -> sandbox -> TASK_MODE=explain_from_result
    - qualitative: TASK_MODE=qualitative_answer only, no sandbox
    """
    if eval_kind == "qualitative":
        qualitative_messages = build_qualitative_answer_messages(problem_id, question)
        try:
            answer_json, answer_response = generate_json(qualitative_messages, max_new_tokens=1024)
        except Exception as e:
            answer_response = ""
            answer_json = {"error": f"Failed to parse qualitative answer JSON: {e}"}

        return {
            "problem_id": problem_id,
            "question": question,
            "eval_kind": eval_kind,
            "generated_code": "",
            "raw_code_response": "",
            "code_json": {},
            "sandbox_result": {"status": "not_applicable", "result": {}},
            "explanation": answer_json,
            "raw_explanation_response": answer_response,
        }

    # Step 1: Generate code using explicit task mode.
    code_messages = build_generate_code_messages(problem_id, question)
    try:
        code_json, code_response = generate_json(code_messages, max_new_tokens=1024)
        generated_code = code_json.get("code", "")
    except Exception as e:
        code_response = ""
        generated_code = ""
        code_json = {"error": f"Failed to parse code JSON: {e}"}

    # Step 2: Execute in sandbox.
    sandbox_result = run_sandbox(generated_code)

    # Step 3: Explain from result using explicit task mode.
    runner_result_for_prompt = sandbox_result["result"] if sandbox_result["status"] == "ok" else sandbox_result
    explain_messages = build_explain_from_result_messages(problem_id, question, runner_result_for_prompt)
    try:
        explain_json, explain_response = generate_json(explain_messages, max_new_tokens=1024)
    except Exception as e:
        explain_response = ""
        explain_json = {"error": f"Failed to parse explanation JSON: {e}"}

    return {
        "problem_id": problem_id,
        "question": question,
        "eval_kind": eval_kind,
        "generated_code": generated_code,
        "raw_code_response": code_response,
        "code_json": code_json,
        "sandbox_result": sandbox_result,
        "explanation": explain_json,
        "raw_explanation_response": explain_response,
    }

print("Option-B evaluation pipeline ready.")


## Secure LLM Judge Configuration

Run this before the evaluation cells if you want the LLM judge to compare model answers with gold answers. The API key is entered through a hidden input field, similar to the HuggingFace token cell.


In [ ]:
# =========================================================================
# 🔐 Secure LLM Judge Config Input
# =========================================================================
# This mirrors the HF token cell:
# - API key is read through a hidden prompt.
# - Model and base_url are normal input fields.
# - Values are stored in environment variables for the judge helper below.
# - Leave the API key or model blank to skip LLM judging and use rule-based checks only.

import os
import getpass


def _input_with_default(prompt: str, default: str = "") -> str:
    suffix = f" [{default}]" if default else ""
    value = input(f"{prompt}{suffix}: ").strip()
    return value or default


def configure_llm_judge_interactive():
    current_key = os.environ.get("JUDGE_API_KEY") or os.environ.get("OPENAI_API_KEY") or ""
    current_model = os.environ.get("JUDGE_MODEL", "")
    current_base_url = os.environ.get("JUDGE_BASE_URL", "")

    if current_key:
        print("✓ Existing judge API key found in environment.")
        replace_key = input("Replace judge API key? [y/N]: ").strip().lower() == "y"
        if replace_key:
            judge_api_key = getpass.getpass("Judge API key (hidden): ").strip()
        else:
            judge_api_key = current_key
    else:
        print("No JUDGE_API_KEY / OPENAI_API_KEY found.")
        print("Enter your judge API key, or leave blank to disable the LLM judge.")
        judge_api_key = getpass.getpass("Judge API key (hidden, optional): ").strip()

    judge_model = _input_with_default(
        "Judge model name, e.g. gpt-4.1-mini / gpt-4o-mini / qwen-max / glm-4-plus",
        current_model,
    ).strip()

    judge_base_url = _input_with_default(
        "OpenAI-compatible base_url, optional, e.g. https://api.openai.com/v1 or provider /v1 endpoint",
        current_base_url,
    ).strip()

    if judge_api_key and judge_model:
        os.environ["JUDGE_API_KEY"] = judge_api_key
        os.environ["JUDGE_MODEL"] = judge_model
        if judge_base_url:
            os.environ["JUDGE_BASE_URL"] = judge_base_url
        elif "JUDGE_BASE_URL" in os.environ:
            del os.environ["JUDGE_BASE_URL"]

        print("✓ LLM judge configured.")
        print(f"  Model: {os.environ['JUDGE_MODEL']}")
        print(f"  Base URL: {os.environ.get('JUDGE_BASE_URL', '(provider default)')}")
        print("  API key: hidden")
        return True

    # Keep rule-only eval available without erroring.
    for key in ["JUDGE_API_KEY", "JUDGE_MODEL", "JUDGE_BASE_URL"]:
        if key in os.environ and key != "OPENAI_API_KEY":
            del os.environ[key]
    print("LLM judge not configured. Evaluation will still run with rule-based numeric/unit checks.")
    return False


LLM_JUDGE_ENABLED = configure_llm_judge_interactive()



In [ ]:
# =========================================================================
# Answer Equivalence + LLM Judge
# =========================================================================
# Goal:
# - Catch unit conversions and notation differences before spending judge calls.
# - Use an LLM judge for qualitative/theory answers and ambiguous numeric cases.
# - Never expose the gold answer during generation; judge only after model output.

import hashlib
import time
from typing import Optional, Tuple

# ---- Numeric/unit normalization ----
_UNIT_TABLE = {
    # dimension: voltage
    "v": ("voltage", 1.0), "volt": ("voltage", 1.0), "volts": ("voltage", 1.0),
    "mv": ("voltage", 1e-3), "kv": ("voltage", 1e3),
    # current
    "a": ("current", 1.0), "ampere": ("current", 1.0), "amperes": ("current", 1.0), "amp": ("current", 1.0), "amps": ("current", 1.0),
    "ma": ("current", 1e-3), "ua": ("current", 1e-6), "μa": ("current", 1e-6),
    # resistance
    "ohm": ("resistance", 1.0), "ohms": ("resistance", 1.0), "ω": ("resistance", 1.0), "Ω": ("resistance", 1.0),
    "kohm": ("resistance", 1e3), "kω": ("resistance", 1e3), "kΩ": ("resistance", 1e3),
    "mohm": ("resistance", 1e6), "mω": ("resistance", 1e6), "MΩ": ("resistance", 1e6),
    # capacitance
    "f": ("capacitance", 1.0), "farad": ("capacitance", 1.0), "farads": ("capacitance", 1.0),
    "mf": ("capacitance", 1e-3), "uf": ("capacitance", 1e-6), "μf": ("capacitance", 1e-6),
    "nf": ("capacitance", 1e-9), "pf": ("capacitance", 1e-12),
    # inductance
    "h": ("inductance", 1.0), "henry": ("inductance", 1.0), "henrys": ("inductance", 1.0),
    "mh": ("inductance", 1e-3),
    # charge
    "c": ("charge", 1.0), "coulomb": ("charge", 1.0), "coulombs": ("charge", 1.0),
    "mc": ("charge", 1e-3), "uc": ("charge", 1e-6), "μc": ("charge", 1e-6), "nc": ("charge", 1e-9), "pc": ("charge", 1e-12),
    # force / energy / power
    "n": ("force", 1.0), "newton": ("force", 1.0), "newtons": ("force", 1.0),
    "j": ("energy", 1.0), "joule": ("energy", 1.0), "joules": ("energy", 1.0),
    "w": ("power", 1.0), "watt": ("power", 1.0), "watts": ("power", 1.0), "kw": ("power", 1e3), "mwatt": ("power", 1e-3),
    # frequency
    "hz": ("frequency", 1.0), "khz": ("frequency", 1e3), "mhz": ("frequency", 1e6),
    # electric field
    "v/m": ("electric_field", 1.0), "v m^-1": ("electric_field", 1.0), "n/c": ("electric_field", 1.0),
    # dimensionless / angle
    "": ("dimensionless", 1.0), "unitless": ("dimensionless", 1.0),
    "rad": ("angle", 1.0), "radian": ("angle", 1.0), "radians": ("angle", 1.0),
    "deg": ("angle", math.pi/180), "degree": ("angle", math.pi/180), "degrees": ("angle", math.pi/180),
}

def _normalize_unit_text(unit: str) -> str:
    if unit is None:
        return ""
    u = str(unit).strip()
    u = u.replace("Ω", "Ω").replace("µ", "μ")
    u = u.replace("Ω", "ω")
    u = u.replace("μ", "u")
    u = u.replace(" ", "")
    u = u.replace("−", "-")
    u = u.replace("per", "/")
    u = u.lower()
    # restore common aliases after lowercasing
    aliases = {
        "omega": "ohm",
        "ohm.": "ohm",
        "volts": "v",
        "volt": "v",
        "amperes": "a",
        "ampere": "a",
        "amps": "a",
        "amp": "a",
        "watts": "w",
        "watt": "w",
        "joules": "j",
        "joule": "j",
        "newtons": "n",
        "newton": "n",
        "farads": "f",
        "farad": "f",
        "henrys": "h",
        "henry": "h",
    }
    u = aliases.get(u, u)
    return u

def unit_info(unit: str) -> Optional[Tuple[str, float]]:
    u = _normalize_unit_text(unit)
    return _UNIT_TABLE.get(u)

def parse_first_number(x) -> Optional[float]:
    if x is None:
        return None
    s = str(x).strip()
    if not s:
        return None
    s = s.replace(",", "")
    s = s.replace("−", "-")
    s = s.replace("×", "x")
    s = s.replace("·", "*")
    # Convert forms like 3.2 x 10^-6, 3.2*10^(-6), 3.2e-6
    s = re.sub(
        r"([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*(?:x|\*)\s*10\s*\^?\s*\(?\s*([+-]?\d+)\s*\)?",
        r"\1e\2",
        s,
        flags=re.I,
    )
    m = re.search(r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?", s)
    if not m:
        return None
    try:
        return float(m.group(0))
    except Exception:
        return None

def maybe_extract_unit_from_answer(answer: str) -> str:
    """Extract simple trailing unit when the explicit unit field is empty."""
    if answer is None:
        return ""
    s = str(answer).strip()
    # take trailing non-numeric-ish token
    m = re.search(r"([A-Za-zμµΩΩ/^\-]+)\s*$", s)
    return m.group(1) if m else ""

def rule_based_equivalence(pred_answer, pred_unit, gold_answer, gold_unit, rtol=1e-3, atol=1e-8) -> dict:
    pred_num = parse_first_number(pred_answer)
    gold_num = parse_first_number(gold_answer)

    pred_unit = pred_unit or maybe_extract_unit_from_answer(pred_answer)
    gold_unit = gold_unit or maybe_extract_unit_from_answer(gold_answer)

    pinfo = unit_info(pred_unit)
    ginfo = unit_info(gold_unit)

    out = {
        "source": "rule",
        "equivalent": None,
        "score": 0.0,
        "category": "unknown",
        "reason": "",
        "normalized_pred": "",
        "normalized_gold": "",
    }

    # Numeric + convertible units
    if pred_num is not None and gold_num is not None and pinfo and ginfo:
        pdim, pfactor = pinfo
        gdim, gfactor = ginfo
        if pdim != gdim:
            out.update(
                equivalent=False,
                category="wrong_unit_dimension",
                reason=f"Unit dimensions differ: predicted {pdim}, gold {gdim}.",
            )
            return out

        pred_si = pred_num * pfactor
        gold_si = gold_num * gfactor
        ok = abs(pred_si - gold_si) <= max(atol, rtol * max(abs(gold_si), 1e-12))
        out.update(
            equivalent=bool(ok),
            score=1.0 if ok else 0.0,
            category="numeric_unit_equivalent" if ok else "wrong_numeric_value",
            reason=f"Compared in normalized {pdim} units.",
            normalized_pred=f"{pred_si:g} [{pdim}]",
            normalized_gold=f"{gold_si:g} [{gdim}]",
        )
        return out

    # Numeric only
    if pred_num is not None and gold_num is not None and not gold_unit and not pred_unit:
        ok = abs(pred_num - gold_num) <= max(atol, rtol * max(abs(gold_num), 1e-12))
        out.update(
            equivalent=bool(ok),
            score=1.0 if ok else 0.0,
            category="numeric_equivalent_no_unit" if ok else "wrong_numeric_value_no_unit",
            reason="Compared numeric answers without units.",
            normalized_pred=f"{pred_num:g}",
            normalized_gold=f"{gold_num:g}",
        )
        return out

    out["reason"] = "Rule-based checker could not confidently parse numeric/unit equivalence."
    return out

# ---- OpenAI-compatible LLM judge ----
def get_judge_client():
    """
    Reads the judge config from the secure input cell above.

    Required:
      JUDGE_API_KEY   : provider API key, hidden input field
      JUDGE_MODEL     : judge model name

    Optional:
      JUDGE_BASE_URL  : OpenAI-compatible provider base URL, e.g. provider /v1 endpoint
    """
    api_key = os.environ.get("JUDGE_API_KEY") or os.environ.get("OPENAI_API_KEY")
    model = os.environ.get("JUDGE_MODEL")
    base_url = os.environ.get("JUDGE_BASE_URL")

    if not api_key or not model:
        return None, None

    from openai import OpenAI
    kwargs = {"api_key": api_key}
    if base_url:
        kwargs["base_url"] = base_url
    return OpenAI(**kwargs), model

JUDGE_CACHE_PATH = Path(args.output_dir) / "llm_judge_cache.jsonl"
JUDGE_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

def _judge_cache_key(payload: dict) -> str:
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

def _load_judge_cache() -> dict:
    cache = {}
    if JUDGE_CACHE_PATH.exists():
        with open(JUDGE_CACHE_PATH, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    row = json.loads(line)
                    cache[row["key"]] = row["judgment"]
                except Exception:
                    continue
    return cache

def _append_judge_cache(key: str, judgment: dict) -> None:
    with open(JUDGE_CACHE_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps({"key": key, "judgment": judgment}, ensure_ascii=False) + "\n")

_JUDGE_CACHE = _load_judge_cache()

def llm_judge_equivalence(question, pred_answer, pred_unit, final_answer, gold_answer, gold_unit) -> dict:
    client, judge_model = get_judge_client()
    if client is None:
        return {
            "source": "llm",
            "equivalent": None,
            "score": 0.0,
            "category": "judge_not_configured",
            "reason": "Set JUDGE_API_KEY and JUDGE_MODEL to enable LLM judge.",
            "normalized_pred": "",
            "normalized_gold": "",
        }

    payload = {
        "question": question,
        "pred_answer": pred_answer,
        "pred_unit": pred_unit,
        "final_answer": final_answer,
        "gold_answer": gold_answer,
        "gold_unit": gold_unit,
    }
    key = _judge_cache_key(payload)
    if key in _JUDGE_CACHE:
        cached = dict(_JUDGE_CACHE[key])
        cached["cached"] = True
        return cached

    system = (
        "You are a strict but fair physics answer judge. "
        "Judge whether the model answer is mathematically and physically equivalent to the gold answer. "
        "Accept equivalent unit conversions, scientific notation, rounding within reasonable tolerance, "
        "and equivalent textual/theory answers. Reject wrong dimensions, wrong formulas, sign errors, "
        "and answers that only look similar. Return JSON only."
    )
    user = {
        "question": question,
        "gold_answer": str(gold_answer),
        "gold_unit": str(gold_unit),
        "model_sandbox_answer": str(pred_answer),
        "model_sandbox_unit": str(pred_unit),
        "model_final_answer_text": str(final_answer),
        "required_json_schema": {
            "equivalent": "boolean",
            "score": "number from 0 to 1",
            "category": "correct | equivalent_conversion | minor_rounding | wrong_value | wrong_unit | incomplete | unjudgeable",
            "reason": "brief reason",
            "normalized_pred": "normalized value/unit if applicable",
            "normalized_gold": "normalized value/unit if applicable",
        }
    }

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": json.dumps(user, ensure_ascii=False)},
    ]

    for attempt in range(3):
        try:
            kwargs = dict(model=judge_model, messages=messages, temperature=0)
            try:
                resp = client.chat.completions.create(
                    **kwargs,
                    response_format={"type": "json_object"},
                )
            except TypeError:
                resp = client.chat.completions.create(**kwargs)

            content = resp.choices[0].message.content
            judgment = extract_json_object(content)
            judgment["source"] = "llm"
            judgment["cached"] = False
            _JUDGE_CACHE[key] = judgment
            _append_judge_cache(key, judgment)
            return judgment
        except Exception as e:
            if attempt == 2:
                return {
                    "source": "llm",
                    "equivalent": None,
                    "score": 0.0,
                    "category": "judge_error",
                    "reason": f"LLM judge failed: {e}",
                    "normalized_pred": "",
                    "normalized_gold": "",
                }
            time.sleep(1.5 * (attempt + 1))

def judge_answer(result: dict, use_llm_for_all: bool = False) -> dict:
    """Combine cheap deterministic checker + optional LLM judge."""
    sandbox = result.get("sandbox_result", {})
    sandbox_out = sandbox.get("result", {}) if sandbox.get("status") == "ok" else {}
    if not isinstance(sandbox_out, dict):
        sandbox_out = {}

    pred_answer = sandbox_out.get("answer", "")
    pred_unit = sandbox_out.get("unit", "")
    final_answer = ""
    explanation = result.get("explanation", {})
    if isinstance(explanation, dict):
        final_answer = explanation.get("answer", "")

    # For qualitative_answer tasks, there is no sandbox answer. Judge the final answer text.
    if not pred_answer and final_answer:
        pred_answer = final_answer
        pred_unit = pred_unit or maybe_extract_unit_from_answer(final_answer)

    gold_answer = result.get("gold_answer", "")
    gold_unit = result.get("gold_unit", "")

    rule = rule_based_equivalence(pred_answer, pred_unit, gold_answer, gold_unit)

    # Avoid unnecessary judge calls when numeric+unit equivalence is clear,
    # unless you intentionally want a judge label for every row.
    if rule["equivalent"] is True and not use_llm_for_all:
        return rule

    llm = llm_judge_equivalence(
        question=result.get("question", ""),
        pred_answer=pred_answer,
        pred_unit=pred_unit,
        final_answer=final_answer,
        gold_answer=gold_answer,
        gold_unit=gold_unit,
    )

    # Prefer LLM when configured and decisive; otherwise fall back to rule.
    if llm.get("equivalent") is not None:
        return llm
    return rule

print("Judge helpers ready. Use the secure Judge API configuration cell above to enable LLM judging.")



In [ ]:
# =========================================================================
# Run Held-Out Evaluation + Rule/LLM Judge — Option B
# =========================================================================

eval_model.eval()

# IMPORTANT: eval_lookup was built from held-out problem IDs only.
eval_ids = list(eval_lookup.keys())
import random
random.seed(args.seed)
random.shuffle(eval_ids)

# Set EVAL_LIMIT=0 for full held-out evaluation. Small values are only for smoke tests.
EVAL_LIMIT = int(os.environ.get("EVAL_LIMIT", "0"))
if EVAL_LIMIT > 0:
    eval_ids = eval_ids[:min(EVAL_LIMIT, len(eval_ids))]

USE_LLM_FOR_ALL = os.environ.get("USE_LLM_FOR_ALL", "0") == "1"

print(f"Evaluating on {len(eval_ids)} held-out problems...\n")

results = []
for i, pid in enumerate(eval_ids):
    entry = eval_lookup[pid]
    question = entry["question"]
    eval_kind = entry.get("eval_kind", "computable")
    if not question:
        continue

    print(f"[{i+1}/{len(eval_ids)}] Problem {pid} [{eval_kind}]: {question[:90]}...")
    result = solve_one(question, pid, eval_kind=eval_kind)
    result["gold_answer"] = entry["gold_answer"]
    result["gold_unit"] = entry["gold_unit"]

    judgment = judge_answer(result, use_llm_for_all=USE_LLM_FOR_ALL)
    result["judge"] = judgment
    results.append(result)

    if eval_kind == "computable":
        if result["sandbox_result"]["status"] == "ok":
            print("  Sandbox: OK")
            answer_info = result["sandbox_result"]["result"]
            if isinstance(answer_info, dict):
                print(f"  Computed: {answer_info.get('answer', 'N/A')} {answer_info.get('unit', '')}")
        else:
            print(f"  Sandbox: FAILED - {result['sandbox_result'].get('error', 'unknown')}")
    else:
        print("  Sandbox: N/A qualitative task")
        if isinstance(result.get("explanation"), dict):
            print(f"  Final:    {result['explanation'].get('answer', '')}")

    print(f"  Gold:     {entry['gold_answer']} {entry['gold_unit']}")
    print(f"  Judge:    {judgment.get('equivalent')} | {judgment.get('category')} | {judgment.get('reason', '')[:120]}")
    print()

# ---- Save detailed results ----
result_path = Path(args.output_dir) / "eval_results_option_b_judged.jsonl"
result_path.parent.mkdir(parents=True, exist_ok=True)
with open(result_path, "w", encoding="utf-8") as f:
    for row in results:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# ---- Compute metrics ----
computable_total = 0
sandbox_ok = 0
qualitative_total = 0
judge_correct = 0
json_answer_ok = 0

for r in results:
    if isinstance(r.get("explanation"), dict) and "answer" in r["explanation"]:
        json_answer_ok += 1

    if r.get("eval_kind") == "qualitative":
        qualitative_total += 1
    else:
        computable_total += 1
        if r["sandbox_result"]["status"] == "ok":
            sandbox_ok += 1

    j = r.get("judge", {})
    if j.get("equivalent") is True:
        judge_correct += 1

n = max(len(results), 1)
print(f"{'='*64}")
print("Held-Out Evaluation Results — Option B")
print(f"{'='*64}")
print(f"  Total held-out problems : {len(results)}")
print(f"  Computable problems     : {computable_total}")
print(f"  Qualitative problems    : {qualitative_total}")
print(f"  Answer JSON parse OK    : {json_answer_ok}/{len(results)} ({100*json_answer_ok/n:.1f}%)")
if computable_total > 0:
    print(f"  Sandbox success         : {sandbox_ok}/{computable_total} ({100*sandbox_ok/max(computable_total,1):.1f}%)")
print(f"  Judge-correct answers   : {judge_correct}/{len(results)} ({100*judge_correct/n:.1f}%)")
if sandbox_ok > 0:
    print(f"  Correct of sandbox OK   : {judge_correct}/{sandbox_ok} ({100*judge_correct/max(sandbox_ok,1):.1f}%)")
print(f"  Detailed results saved  : {result_path}")
print(f"{'='*64}")


In [ ]:
# =========================================================================
# Optional: Merge LoRA + Push Full Model to HF Hub
# =========================================================================

MERGE_AND_PUSH = False  # Set to True to merge and push the full model

if MERGE_AND_PUSH:
    merged_hub_id = args.hub_model_id.replace("-lora", "-merged")
    print(f"Merging LoRA weights and pushing to {merged_hub_id}...")
    
    merged_model = eval_model.merge_and_unload()
    merged_model.push_to_hub(merged_hub_id, private=True)
    eval_tokenizer.push_to_hub(merged_hub_id, private=True)
    
    print(f"Merged model pushed to {merged_hub_id}")
else:
    print("Skipping merge. Set MERGE_AND_PUSH=True to push merged model.")